# PubChem Fetch -- Example

This notebook demonstrates `run_pubchem_fetch`, which resolves one of CID, name, SMILES, or InChIKey to canonical small-molecule structure data via the [PubChem PUG REST API](https://pubchem.ncbi.nlm.nih.gov/). Given any single identifier, the tool resolves it to a PubChem CID, fetches a configurable property bundle (formula, weight, SMILES, InChI, descriptors), and optionally returns a list of synonyms. See [Kim et al. 2023](https://doi.org/10.1093/nar/gkac956) for the underlying database.

In [1]:
from proto_tools.tools.database_retrieval import (
    PubChemFetchConfig,
    PubChemFetchInput,
    run_pubchem_fetch,
)

## Input API Reference

Exactly one of the four identifier fields below must be provided.

| Field | Type | Description |
|---|---|---|
| `cid` | `int \| None` | PubChem Compound Identifier (e.g. `2244` for aspirin). |
| `name` | `str \| None` | Common or systematic name (e.g. `"aspirin"`). |
| `smiles` | `str \| None` | SMILES string (e.g. `"CC(=O)Oc1ccccc1C(=O)O"`). |
| `inchikey` | `str \| None` | Standard InChIKey (e.g. `"BSYNRYMUTXBXSQ-UHFFFAOYSA-N"`). |

## Config API Reference

| Field | Type | Default | Description |
|---|---|---|---|
| `properties` | `list[PubChemProperty]` | 15-property bundle (see below) | PubChem property names to request. |
| `include_synonyms` | `bool` | `False` | If True, fetch the compound's synonyms list (up to 50 returned). |
| `include_description` | `bool` | `False` | If True, fetch the compound's textual descriptions (one extra HTTP call). |
| `include_aids` | `bool` | `False` | If True, fetch the list of BioAssay IDs that tested this compound (can return thousands of IDs). |

The default `properties` bundle covers structure, mass, and basic descriptor counts: `MolecularFormula`, `MolecularWeight`, `SMILES`, `ConnectivitySMILES`, `InChI`, `InChIKey`, `IUPACName`, `ExactMass`, `TPSA`, `Complexity`, `Charge`, `HBondDonorCount`, `HBondAcceptorCount`, `RotatableBondCount`, `HeavyAtomCount`.

## Output API Reference

| Field | Type | Description |
|---|---|---|
| `cid` | `int` | Resolved PubChem CID. |
| `all_matched_cids` | `list[int]` | All CIDs returned by the resolver (length 1 for unambiguous queries; may be longer for ambiguous names). |
| `molecular_formula` | `str \| None` | Hill-system molecular formula. |
| `molecular_weight` | `float \| None` | Average molecular weight (g/mol). |
| `smiles` | `str \| None` | PubChem canonical SMILES with stereochemistry preserved (formerly `IsomericSMILES`). |
| `connectivity_smiles` | `str \| None` | Connectivity-only SMILES, with stereochemistry stripped (formerly `CanonicalSMILES`). |
| `inchi` | `str \| None` | Standard InChI string. |
| `inchikey` | `str \| None` | Standard InChIKey hash. |
| `iupac_name` | `str \| None` | IUPAC systematic name. |
| `exact_mass` | `float \| None` | Exact (monoisotopic) mass in Da. |
| `tpsa` | `float \| None` | Topological polar surface area, in angstroms-squared. |
| `complexity` | `int \| None` | Bertz / Hendrickson / Ihlenfeldt complexity. |
| `charge` | `int \| None` | Net formal charge. |
| `hbond_donor_count` | `int \| None` | Number of hydrogen-bond donors. |
| `hbond_acceptor_count` | `int \| None` | Number of hydrogen-bond acceptors. |
| `rotatable_bond_count` | `int \| None` | Number of rotatable bonds. |
| `heavy_atom_count` | `int \| None` | Number of non-hydrogen atoms. |
| `synonyms` | `list[str]` | Up to 50 synonyms (empty when `include_synonyms` is False). |
| `descriptions` | `list[str]` | Textual descriptions, one per source (empty when `include_description` is False). |
| `bioassay_ids` | `list[int]` | BioAssay IDs that tested this compound (empty when `include_aids` is False). |
| `source_url` | `str` | URL of the PubChem property request. |
| `raw_property_record` | `dict[str, Any]` | Complete property record from PubChem for advanced programmatic access. |

## Lookup by name

The most common starting point: pass a common name and let PubChem resolve it. Here we fetch aspirin and inspect the canonical structure plus a few descriptors.

In [2]:
inputs = PubChemFetchInput(name="aspirin")
config = PubChemFetchConfig()
output = run_pubchem_fetch(inputs, config)

print(f"CID: {output.cid}")
print(f"Molecular formula: {output.molecular_formula}")
print(f"Molecular weight: {output.molecular_weight}")
print(f"SMILES: {output.smiles}")
print(f"InChIKey: {output.inchikey}")
print(f"IUPAC name: {output.iupac_name}")
print(f"TPSA: {output.tpsa}")
print(f"H-bond donors: {output.hbond_donor_count}")
print(f"H-bond acceptors: {output.hbond_acceptor_count}")

CID: 2244
Molecular formula: C9H8O4
Molecular weight: 180.16
SMILES: CC(=O)OC1=CC=CC=C1C(=O)O
InChIKey: BSYNRYMUTXBXSQ-UHFFFAOYSA-N
IUPAC name: 2-acetyloxybenzoic acid
TPSA: 63.6
H-bond donors: 1
H-bond acceptors: 4


## Lookup by CID

When you already know the PubChem CID, pass it directly to skip the resolver step. Here we fetch caffeine (CID 2519).

In [3]:
caffeine_output = run_pubchem_fetch(PubChemFetchInput(cid=2519), PubChemFetchConfig())

print(f"CID: {caffeine_output.cid}")
print(f"Molecular formula: {caffeine_output.molecular_formula}")
print(f"Molecular weight: {caffeine_output.molecular_weight}")
print(f"SMILES: {caffeine_output.smiles}")
print(f"IUPAC name: {caffeine_output.iupac_name}")

CID: 2519
Molecular formula: C8H10N4O2
Molecular weight: 194.19
SMILES: CN1C=NC2=C1C(=O)N(C(=O)N2C)C
IUPAC name: 1,3,7-trimethylpurine-2,6-dione


## Lookup by SMILES

PubChem can resolve a SMILES string back to its registered compound record. Here we round-trip aspirin's SMILES and verify the resolved CID.

In [4]:
smiles_output = run_pubchem_fetch(
    PubChemFetchInput(smiles="CC(=O)Oc1ccccc1C(=O)O"),
    PubChemFetchConfig(),
)

print(f"Resolved CID: {smiles_output.cid}")
print(f"Molecular formula: {smiles_output.molecular_formula}")
assert smiles_output.cid == 2244, f"Expected CID 2244 for aspirin, got {smiles_output.cid}"

Resolved CID: 2244
Molecular formula: C9H8O4


## Lookup by InChIKey

InChIKey is a fixed-length hash of the InChI string and is the most reliable identifier for exact-structure matching. Here we look up aspirin again via its InChIKey.

In [5]:
inchikey_output = run_pubchem_fetch(
    PubChemFetchInput(inchikey="BSYNRYMUTXBXSQ-UHFFFAOYSA-N"),
    PubChemFetchConfig(),
)

print(f"Resolved CID: {inchikey_output.cid}")
print(f"IUPAC name: {inchikey_output.iupac_name}")
assert inchikey_output.cid == 2244, f"Expected CID 2244 for aspirin, got {inchikey_output.cid}"

Resolved CID: 2244
IUPAC name: 2-acetyloxybenzoic acid


## Fetch with synonyms

Setting `include_synonyms=True` issues an extra HTTP request for the compound's synonym list. PubChem can return hundreds of synonyms per compound; the wrapper caps the returned list at 50. This is handy when you need to map between trade names, brand names, and chemical identifiers.

In [ ]:
synonyms_output = run_pubchem_fetch(
    PubChemFetchInput(name="aspirin"),
    PubChemFetchConfig(include_synonyms=True),
)

print(f"CID: {synonyms_output.cid}")
print(f"Synonyms ({len(synonyms_output.synonyms)} of up to 50):")
for synonym in synonyms_output.synonyms[:10]:
    print(f"  - {synonym}")

## Custom property subset

When you only need a few fields, restrict the `properties` list to keep the response small. Here we ask only for formula, weight, and SMILES.

In [7]:
subset_output = run_pubchem_fetch(
    PubChemFetchInput(name="aspirin"),
    PubChemFetchConfig(properties=["MolecularFormula", "MolecularWeight", "SMILES"]),
)

print(f"Molecular formula: {subset_output.molecular_formula}")
print(f"Molecular weight: {subset_output.molecular_weight}")
print(f"SMILES: {subset_output.smiles}")

Molecular formula: C9H8O4
Molecular weight: 180.16
SMILES: CC(=O)OC1=CC=CC=C1C(=O)O


## Export Results

Fetched results can be saved to JSON format for downstream use in other tools or analysis pipelines.

In [8]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

output.export("aspirin_pubchem", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'aspirin_pubchem.json'}")

Exported to example_output/aspirin_pubchem.json
